# 03 · Xem lớp mạng và can thiệp

Đọc trạng thái thật từ checkpoint; không dùng animation mô phỏng việc học.

## Mở trên Colab hoặc Jupyter

Đây là notebook hướng dẫn tái lập, các cell chưa được đánh dấu là đã thực thi. Kết quả huấn luyện thật có sẵn trong `artifacts/`.

**Colab:** tải ZIP đầy đủ lên Files, giải nén vào `/content/` để có `/content/Neural_Observatory`. Có thể dùng `import zipfile; zipfile.ZipFile('/content/Neural_Observatory.zip').extractall('/content')` với ZIP được bàn giao. Chọn CPU; bật T4 không tăng tốc NumPy tự động. Nếu cần dependency, chạy `%pip install -r /content/Neural_Observatory/requirements.txt`.

**Jupyter:** mở notebook từ `notebooks/` sau khi cài `requirements.txt`. Chạy các cell từ trên xuống.


In [ ]:
from pathlib import Path
import os, sys, subprocess
# Local: launch notebook from the extracted project or its notebooks/ folder.
# Colab: extract Neural_Observatory.zip into /content first (see instructions).
candidates = [Path.cwd(), Path.cwd().parent, Path('/content/Neural_Observatory')]
project = next((p for p in candidates if (p/'observatory/engine.py').exists()), None)
assert project is not None, 'Extract the full project ZIP first and set project to its root.'
os.chdir(project)
sys.path.insert(0, str(project))
print('Project:', project)


In [ ]:
from observatory.dataset import load_numpy
from observatory.engine import MLP
x,y,xv,yv,xt,yt,_,_=load_numpy()
m=MLP.load('artifacts/checkpoints/observatory_64_32_s42/best.npz')
a,z=m.forward(xt[:1])
for i,v in enumerate(a):
    print('Layer',i,'shape',v.shape,'range',float(v.min()),float(v.max()))
print('Truth:',yt[0],'Prediction:',a[-1].argmax())

## Activation và weight maps

Weight map của lớp đầu có cấu trúc không gian của ảnh; lưới của lớp sâu hơn chỉ là cách sắp neuron. Activation = 0 ở một ảnh chưa phải neuron chết.

In [ ]:
import matplotlib.pyplot as plt
fig,axs=plt.subplots(1,4,figsize=(12,3))
axs[0].imshow(xt[0].reshape(28,28),cmap='gray');axs[0].set_title('Input')
axs[1].imshow(a[1].reshape(8,8),cmap='Purples');axs[1].set_title('Hidden 1: 64')
axs[2].imshow(a[2].reshape(4,8),cmap='Purples');axs[2].set_title('Hidden 2: 32')
axs[3].bar(range(10),a[-1][0]);axs[3].set_title('Softmax')
plt.tight_layout();plt.show()

## Can thiệp neuron

Đặt activation một neuron bằng 0, giữ các trọng số còn lại. Xem thay đổi trên cùng ảnh, rồi trên validation. Không gán ý nghĩa nhân quả thế giới thực cho thao tác này.

In [ ]:
before=m.forward(xt[:1])[0][-1][0]
after=m.forward(xt[:1],intervention=(1,11))[0][-1][0]
print('Before:',before,'After:',after)
print('Pred:',before.argmax(),'->',after.argmax())

## Mở toàn bộ observatory

File HTML offline chứa trọng số thật. Nếu đã chạy lại thí nghiệm, build lại trước. Trên Colab có thể tải HTML về và mở bằng trình duyệt.

In [ ]:
REBUILD=False
if REBUILD:
    subprocess.run([sys.executable,'main.py','build'],check=True)
print(Path('artifacts/Neural_Observatory.html').resolve())

## Điều cần báo cáo

Hãy mở ảnh sai, INT8 đổi dự đoán, ảnh bị dịch và latency chậm hơn. Không chỉ chọn những ảnh mạng làm đúng.